In [1]:
import polars as pl

In [2]:
pl.Config.set_tbl_cols(-1)  # Display all columns
pl.Config.set_tbl_rows(-1)  # Display all rows

polars.config.Config

In [3]:
#script to create a metada file for wagtail run
#inside the metadata file, we will have the following columns:
#1. sample_id
#2. total input reads (from qc reports)
#3. total retained reads (from qc reports)
#4. total retained reads percentage (#3/#2*100)
#5-18. deblur stats result
#19. total reads for mapping (from samtools flagstat reports line 1)
#20. total all mapped reads with percentage (from samtools flagstat reports line 7)
#21. total all primary mapped reads with percentage (from samtools flagstat reports line 8)

#1-18 is combining from the qc-stats report with deblur stats result
#19-21 is from samtools flagstat report and take only line 1, 7, and 8

In [3]:
#read the qc-stats report and deblur stats result
qc_stats = pl.read_csv("qc-stats.csv")
deblur_stats = pl.read_csv("deblur-stats.csv")


In [4]:
qc_stats[:4]

sample-id,total-input-reads,total-retained-reads,reads-truncated,reads-too-short-after-truncation,reads-exceeding-maximum-ambiguous-bases
str,i64,i64,i64,i64,i64
"""V1V2""",73176,73031,104,104,41


In [5]:
#add percentage of total retained reads to the qc_stats dataframe
#calculate the total retained reads/total input reads * 100 and save it to a new column called retained-reads-percentage
qc_stats1 = qc_stats.with_columns((pl.col("total-retained-reads") / pl.col("total-input-reads") * 100).alias("retained-reads-percentage"))
qc_stats1[:4]

sample-id,total-input-reads,total-retained-reads,reads-truncated,reads-too-short-after-truncation,reads-exceeding-maximum-ambiguous-bases,retained-reads-percentage
str,i64,i64,i64,i64,i64,f64
"""V1V2""",73176,73031,104,104,41,99.801848


In [6]:
deblur_stats[:4]

sample-id,reads-raw,unique-reads-derep,reads-derep,unique-reads-deblur,reads-deblur,unique-reads-hit-artifact,reads-hit-artifact,unique-reads-chimeric,reads-chimeric,unique-reads-hit-reference,reads-hit-reference,unique-reads-missed-reference,reads-missed-reference
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""V1V2""",73031,2992,51264,174,13683,0,0,31,607,143,13076,0,0


In [9]:
#combine the qc_stats1 and deblur_stats dataframe
#we will use the sample-id column to join the two dataframes
metadata = qc_stats1.join(deblur_stats, on="sample-id")
metadata[:4]

sample-id,total-input-reads,total-retained-reads,reads-truncated,reads-too-short-after-truncation,reads-exceeding-maximum-ambiguous-bases,retained-reads-percentage,reads-raw,unique-reads-derep,reads-derep,unique-reads-deblur,reads-deblur,unique-reads-hit-artifact,reads-hit-artifact,unique-reads-chimeric,reads-chimeric,unique-reads-hit-reference,reads-hit-reference,unique-reads-missed-reference,reads-missed-reference
str,i64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""V1V2""",73176,73031,104,104,41,99.801848,73031,2992,51264,174,13683,0,0,31,607,143,13076,0,0


In [12]:
#read the samtools flagstat report
metadata2 = pl.read_csv("V1V2_2_metadata.tsv", separator= '\t', has_header=True)
metadata2[:4]

sample-id,total reads,mapped reads,unmapped reads,mapping percentage,average mapq
str,i64,i64,i64,f64,f64
"""V1V2""",129,129,0,100.0,0.356589


In [13]:
#combine the metadata and metadata2 dataframe
#we will use the sample-id column to join the two dataframes
metadata3 = metadata.join(metadata2, on="sample-id")
metadata3[:4]

sample-id,total-input-reads,total-retained-reads,reads-truncated,reads-too-short-after-truncation,reads-exceeding-maximum-ambiguous-bases,retained-reads-percentage,reads-raw,unique-reads-derep,reads-derep,unique-reads-deblur,reads-deblur,unique-reads-hit-artifact,reads-hit-artifact,unique-reads-chimeric,reads-chimeric,unique-reads-hit-reference,reads-hit-reference,unique-reads-missed-reference,reads-missed-reference,total reads,mapped reads,unmapped reads,mapping percentage,average mapq
str,i64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64
"""V1V2""",73176,73031,104,104,41,99.801848,73031,2992,51264,174,13683,0,0,31,607,143,13076,0,0,129,129,0,100.0,0.356589


In [14]:
#read some file and then append the metadata1 dataframe to the file
metadata3.write_csv("metadata_combined-mappy.tsv", separator="\t", has_header=True)